# 02 — LSTM & RNN Direct Sector Prediction
**Authors: Marcus Hau, Sun Cho**

This notebook implements Pipeline 2: instead of predicting a macro regime
and investing in historically best sectors, we directly predict whether each
sector will go UP or DOWN next month using **sequence models** that read the
last 12 months of macroeconomic indicators: an **LSTM** and a **vanilla RNN**
(Elman RNN) for comparison.

We then compare these approaches against the macro regime strategy
from notebook 01 and SPY as benchmark.

**Requires:** Run 01_EDA_and_Regimes.ipynb first and save outputs to Drive.


In [ ]:
# ── CELL 1: Install packages ───────────────────────────────────
!pip install torch yfinance scikit-learn xgboost --quiet
print('✓ Done')

In [ ]:
# ── CELL 2: Mount Google Drive ─────────────────────────────────
from google.colab import drive
import os

drive.mount('/content/drive')

DATA_DIR    = '/content/drive/MyDrive/CDS DS340/Project/data'
RESULTS_DIR = '/content/drive/MyDrive/CDS DS340/Project/results'
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f'✓ Drive mounted')
print(f'  Data:    {DATA_DIR}')
print(f'  Results: {RESULTS_DIR}')

In [ ]:
# ── CELL 3: Imports and config ─────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mtick
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
import xgboost as xgb

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if device.type == 'cuda':
    print(f'  GPU: {torch.cuda.get_device_name(0)}')

# ── Config (must match notebook 01 exactly) ────────────────────
SECTOR_ETFS = {
    'XLK':'Information Technology', 'XLF':'Financials',
    'XLE':'Energy',                 'XLV':'Health Care',
    'XLI':'Industrials',            'XLP':'Consumer Staples',
    'XLY':'Consumer Discretionary', 'XLU':'Utilities',
    'XLB':'Materials',              'XLRE':'Real Estate',
    'XLC':'Communication Services',
}
BENCHMARK     = 'SPY'
FEATURE_COLS  = [
    'yield_curve', 'ism_pmi', 'consumer_conf',
    'jobless_claims_yoy', 'lei', 'gdp_real_yoy',
    'cpi_yoy', 'unemployment',
]
LAGS = {'gdp_real': 3, 'cpi': 1, 'unemployment': 1}
REGIME_COLORS = {
    'Goldilocks':        '#2ecc71',
    'Reflation':         '#f39c12',
    'Stagflation':       '#e74c3c',
    'Deflationary Bust': '#3498db',
}

TOP_N_SECTORS    = 3
TRANSACTION_COST = 0.001
RISK_FREE_RATE   = 0.04
TEST_YEARS       = 5
RANDOM_STATE     = 42

# LSTM hyperparameters
LSTM_WINDOW  = 12   # months of history fed to LSTM as one input sequence
LSTM_HIDDEN  = 64   # hidden state size
LSTM_LAYERS  = 2    # stacked LSTM layers
LSTM_DROPOUT = 0.2
LSTM_EPOCHS  = 100
LSTM_LR      = 0.001
LSTM_BATCH   = 32

print('✓ Config set')
print(f'  LSTM window: {LSTM_WINDOW} months')
print(f'  LSTM hidden: {LSTM_HIDDEN}, layers: {LSTM_LAYERS}')
print(f'  Epochs: {LSTM_EPOCHS}, LR: {LSTM_LR}')

# Vanilla RNN uses the same window / epochs / LR / batch as LSTM for a fair comparison.


In [ ]:
# ── CELL 4: Load all data from Drive ──────────────────────────
# Load raw data
macro_df   = pd.read_csv(f'{DATA_DIR}/macro_raw.csv',      index_col=0, parse_dates=True)
returns_df = pd.read_csv(f'{DATA_DIR}/sector_returns.csv', index_col=0, parse_dates=True)

# Load regime labels saved from notebook 01
# NOTE: add this to the END of notebook 01 first:
#   regime_labels.to_csv(f'{DATA_DIR}/regime_labels.csv')
regime_labels = pd.read_csv(f'{DATA_DIR}/regime_labels.csv',
                             index_col=0, parse_dates=True).squeeze()

macro_df['gdp_real'] = macro_df['gdp_real'].ffill()
sector_cols = [c for c in returns_df.columns if c != BENCHMARK]

print(f'✓ Macro data:     {macro_df.shape}')
print(f'✓ Returns:        {returns_df.shape}')
print(f'✓ Regime labels:  {len(regime_labels)} months')
print(f'  Regime distribution:')
print(regime_labels.value_counts().to_string())

In [ ]:
# ── CELL 5: Feature engineering (identical to notebook 01) ────
def apply_lags(df):
    df = df.copy()
    for col, lag in LAGS.items():
        if col in df.columns:
            df[col] = df[col].shift(lag)
    return df

features_df = apply_lags(macro_df.copy())
features_df['gdp_real_yoy']       = features_df['gdp_real'].pct_change(12) * 100
features_df['cpi_yoy']            = features_df['cpi'].pct_change(12) * 100
features_df['jobless_claims_yoy'] = features_df['jobless_claims'].pct_change(12) * 100
nber = features_df['nber_recession'].copy() if 'nber_recession' in features_df.columns else None

avail_cols = [f for f in FEATURE_COLS if f in features_df.columns]
X_raw      = features_df[avail_cols].dropna()

# Align: features at T predict returns at T+1
returns_next = returns_df[sector_cols].shift(-1)
common       = X_raw.index.intersection(returns_next.dropna(how='all').index)
X            = X_raw.loc[common].dropna()
y_returns    = returns_next.loc[X.index].dropna(how='all')
bm_returns   = returns_df[BENCHMARK].reindex(X.index)

valid      = X.notna().all(axis=1) & y_returns.notna().any(axis=1)
X          = X[valid]
y_returns  = y_returns[valid]
bm_returns = bm_returns[valid]

# Temporal train/test split
n       = len(X)
test_n  = TEST_YEARS * 12
train_n = n - test_n

X_train   = X.iloc[:train_n];  X_test  = X.iloc[train_n:]
y_train   = y_returns.iloc[:train_n]; y_test = y_returns.iloc[train_n:]
bm_train  = bm_returns.iloc[:train_n]; bm_test = bm_returns.iloc[train_n:]

# Normalize — fit on training only
scaler    = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)
X_all_s   = scaler.transform(X)

print(f'✓ Feature matrix: {X.shape}')
print(f'  Features: {avail_cols}')
print(f'  Train: {X_train.index[0].date()} → {X_train.index[-1].date()} ({len(X_train)} months)')
print(f'  Test:  {X_test.index[0].date()} → {X_test.index[-1].date()} ({len(X_test)} months)')

In [ ]:
# ── CELL 6: Binary direction labels ───────────────────────────
# Target variable: 1 = sector goes UP next month, 0 = goes DOWN
# This is what the LSTM predicts
y_dir = (y_returns > 0).astype(int)

print('=== Direction Label Statistics ===')
print(f'  (1 = sector up next month, 0 = sector down)')
print()
print(f'{"Sector":<8} {"Name":<30} {"Up%":>6} {"Down%":>6} {"Majority Baseline":>18}')
print('-' * 70)
for ticker in sector_cols:
    if ticker not in y_dir.columns:
        continue
    up_pct   = y_dir[ticker].mean() * 100
    baseline = max(y_dir[ticker].mean(), 1 - y_dir[ticker].mean())
    print(f'{ticker:<8} {SECTOR_ETFS.get(ticker,""):<30} '
          f'{up_pct:>6.1f}% {100-up_pct:>6.1f}% {baseline:>18.3f}')

print()
print('Majority baseline = accuracy you get by always predicting the most common outcome')
print('Our models need to beat this to be useful')

In [ ]:
# ── CELL 7: LSTM architecture ──────────────────────────────────
class SectorLSTM(nn.Module):
    """
    LSTM for binary sector direction prediction.

    Input shape:  (batch_size, sequence_length=12, n_features=8)
    Output shape: (batch_size,)  probability sector goes UP next month

    How it works:
      1. LSTM reads 12 months of macro data one month at a time
      2. At each step it updates a hidden state (memory)
      3. After reading all 12 months, the final hidden state
         is a compressed summary of recent economic history
      4. A small classifier converts that summary to a probability

    Why LSTM and not a simple neural network:
      A simple NN treats each month independently.
      LSTM captures patterns ACROSS months — e.g. 'yield curve
      has been inverting for 3 consecutive months while jobless
      claims are rising' — temporal patterns a static model misses.
    """
    def __init__(self, n_features, hidden=LSTM_HIDDEN,
                 n_layers=LSTM_LAYERS, dropout=LSTM_DROPOUT):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=n_features,
            hidden_size=hidden,
            num_layers=n_layers,
            batch_first=True,          # input: (batch, seq, features)
            dropout=dropout if n_layers > 1 else 0
        )

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1),
            nn.Sigmoid()               # output: probability 0-1
        )

    def forward(self, x):
        # x: (batch, 12, 8)
        _, (h_n, _) = self.lstm(x)
        # h_n[-1]: final hidden state (batch, hidden)
        return self.classifier(h_n[-1]).squeeze(-1)


class SectorRNN(nn.Module):
    """Vanilla Elman RNN + same classifier head as SectorLSTM (fair comparison)."""
    def __init__(self, n_features, hidden=LSTM_HIDDEN,
                 n_layers=LSTM_LAYERS, dropout=LSTM_DROPOUT, nonlinearity='tanh'):
        super().__init__()
        self.rnn = nn.RNN(
            input_size=n_features,
            hidden_size=hidden,
            num_layers=n_layers,
            batch_first=True,
            nonlinearity=nonlinearity,
            dropout=dropout if n_layers > 1 else 0,
        )
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        _, h_n = self.rnn(x)
        return self.classifier(h_n[-1]).squeeze(-1)


for Cls, name in [(SectorLSTM, 'LSTM'), (SectorRNN, 'RNN')]:
    test_model  = Cls(n_features=len(avail_cols)).to(device)
    test_input  = torch.randn(4, LSTM_WINDOW, len(avail_cols)).to(device)
    test_output = test_model(test_input)
    total_params = sum(p.numel() for p in test_model.parameters())
    print(f'✓ {name} OK  params={total_params:,}  out={list(test_output.shape)}  '
          f'range {test_output.min().item():.3f}..{test_output.max().item():.3f}')
    del test_model, test_input, test_output


In [ ]:
# ── CELL 8: Sliding window sequence builder ────────────────────
def make_sequences(X_arr, y_arr, window=LSTM_WINDOW):
    """
    Convert flat arrays into sliding window sequences for LSTM.

    For each month T (where T >= window):
      Input  = X[T-window : T]  (12 months of features, shape: window x features)
      Target = y[T]             (did sector go up in month T?)

    Example with window=3:
      Month 3: input=[Jan,Feb,Mar], target=Apr direction
      Month 4: input=[Feb,Mar,Apr], target=May direction
      Month 5: input=[Mar,Apr,May], target=Jun direction
      ...window slides forward one month at a time

    This is the 'sliding window training' the TA mentioned.
    Each training example overlaps with the previous one by window-1 months.
    """
    Xs, ys = [], []
    for i in range(len(X_arr) - window):
        Xs.append(X_arr[i : i + window])  # shape: (window, n_features)
        ys.append(y_arr[i + window])       # scalar target
    return np.array(Xs), np.array(ys)

# Verify
test_X, test_y = make_sequences(X_all_s, y_dir.iloc[:,0].values)
print(f'✓ Sequence builder works')
print(f'  Input  shape: {test_X.shape}  (n_sequences, window, n_features)')
print(f'  Target shape: {test_y.shape}  (n_sequences,)')
print(f'  Total sequences from {len(X)} months: {len(test_X)}')
print(f'  (lost {LSTM_WINDOW} months at start for the first window)')
del test_X, test_y

In [ ]:
# ── CELL 9: Train LSTM and vanilla RNN per sector ─────────────
# Same sliding windows, splits, and optimization recipe for both.

def train_sequence_models_per_sector(ModelClass, label):
    results = {}
    seq_train_end = train_n - LSTM_WINDOW

    print(f'=== {label} Training — One Model Per Sector ===')
    print(f'Window: {LSTM_WINDOW} months | Epochs: {LSTM_EPOCHS} | Device: {device}')
    print(f'Train sequences end at index: {seq_train_end}')
    print()

    for ticker in sector_cols:
        print(f'━━━ {ticker} ({SECTOR_ETFS.get(ticker, "")}) ━━━')

        y_arr      = y_dir[ticker].values.astype(float)
        Xs, ys     = make_sequences(X_all_s, y_arr, LSTM_WINDOW)

        Xtr = Xs[:seq_train_end];  ytr = ys[:seq_train_end]
        Xte = Xs[seq_train_end:];  yte = ys[seq_train_end:]

        vtr = ~np.isnan(ytr);  vte = ~np.isnan(yte)
        Xtr, ytr = Xtr[vtr], ytr[vtr]
        Xte, yte = Xte[vte], yte[vte]

        if len(Xtr) < 30:
            print(f'  Skipping — only {len(Xtr)} training sequences\n')
            continue

        val_n      = max(int(len(Xtr) * 0.15), 10)
        Xval, yval = Xtr[-val_n:], ytr[-val_n:]
        Xtr,  ytr  = Xtr[:-val_n], ytr[:-val_n]

        print(f'  Train: {len(Xtr)} seq | Val: {len(Xval)} seq | Test: {len(Xte)} seq')

        Xt    = torch.FloatTensor(Xtr).to(device)
        yt    = torch.FloatTensor(ytr).to(device)
        Xv    = torch.FloatTensor(Xval).to(device)
        yv    = torch.FloatTensor(yval).to(device)
        Xte_t = torch.FloatTensor(Xte).to(device)

        loader = DataLoader(TensorDataset(Xt, yt),
                            batch_size=LSTM_BATCH, shuffle=False)

        model = ModelClass(n_features=X_all_s.shape[1]).to(device)
        opt   = optim.Adam(model.parameters(), lr=LSTM_LR, weight_decay=1e-4)
        crit  = nn.BCELoss()
        sched = optim.lr_scheduler.ReduceLROnPlateau(
            opt, patience=15, factor=0.5
        )

        train_losses = []
        val_accs     = []
        best_acc     = 0
        best_state   = None

        for epoch in range(LSTM_EPOCHS):
            model.train()
            ep_loss = 0
            for xb, yb in loader:
                opt.zero_grad()
                loss = crit(model(xb), yb)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
                ep_loss += loss.item()

            model.eval()
            with torch.no_grad():
                val_pred = (model(Xv) >= 0.5).float()
                val_acc  = (val_pred == yv).float().mean().item()

            avg_loss = ep_loss / len(loader)
            train_losses.append(avg_loss)
            val_accs.append(val_acc)
            sched.step(avg_loss)

            if val_acc > best_acc:
                best_acc   = val_acc
                best_state = {k: v.clone() for k, v in model.state_dict().items()}

            if (epoch + 1) % 25 == 0:
                print(f'  Epoch {epoch+1:>3}/{LSTM_EPOCHS}: '
                      f'loss={avg_loss:.4f}  val_acc={val_acc:.3f}  '
                      f'best={best_acc:.3f}')

        model.load_state_dict(best_state)
        model.eval()

        with torch.no_grad():
            probs = model(Xte_t).cpu().numpy()
            preds = (probs >= 0.5).astype(int)

        acc      = accuracy_score(yte.astype(int), preds)
        baseline = max(ytr.mean(), 1 - ytr.mean())
        beat     = '✓ BEATS BASELINE' if acc > baseline else '✗ below baseline'

        seq_dates = X.index[seq_train_end + LSTM_WINDOW:]
        seq_dates = seq_dates[vte][:len(preds)]

        results[ticker] = {
            'model':         model,
            'predictions':   preds,
            'probabilities': probs,
            'accuracy':      acc,
            'baseline':      baseline,
            'best_val_acc':  best_acc,
            'dates':         seq_dates,
            'y_true':        yte.astype(int),
            'train_losses':  train_losses,
            'val_accs':      val_accs,
        }

        print(f'  Test accuracy: {acc:.3f}  Baseline: {baseline:.3f}  {beat}')
        print()

    print(f'\n=== {label} Training Complete ===')
    print(f'{"Sector":<8} {"Acc":>10} {"Baseline":>10} {"Beats?":>8}')
    print('-' * 40)
    for tkr, res in results.items():
        beat = '✓' if res['accuracy'] > res['baseline'] else '✗'
        print(f'{tkr:<8} {res["accuracy"]:>10.3f} '
              f'{res["baseline"]:>10.3f} {beat:>8}')
    return results


lstm_results = train_sequence_models_per_sector(SectorLSTM, 'LSTM')
rnn_results  = train_sequence_models_per_sector(SectorRNN, 'Vanilla RNN')


In [ ]:
# ── CELL 10: Plot training curves ─────────────────────────────
n_sectors = len(lstm_results)
n_cols    = 4
n_rows    = int(np.ceil(n_sectors / n_cols))

fig, axes = plt.subplots(n_rows * 2, n_cols,
                          figsize=(n_cols * 5, n_rows * 8))

for i, (ticker, res) in enumerate(lstm_results.items()):
    row_loss = (i // n_cols) * 2
    row_acc  = row_loss + 1
    col      = i % n_cols

    # Training loss
    ax1 = axes[row_loss, col]
    ax1.plot(res['train_losses'], color='#e74c3c', linewidth=1.5)
    ax1.set_title(f'{ticker} — {SECTOR_ETFS.get(ticker,"")}',
                  fontsize=9, fontweight='bold')
    ax1.set_ylabel('Training Loss', fontsize=8)
    ax1.set_xlabel('Epoch', fontsize=8)
    ax1.grid(True, alpha=0.3)
    ax1.tick_params(labelsize=7)

    # Validation accuracy
    ax2 = axes[row_acc, col]
    ax2.plot(res['val_accs'], color='#3498db', linewidth=1.5)
    ax2.axhline(res['baseline'], color='#e74c3c',
                linestyle='--', linewidth=1.2,
                label=f'Baseline {res["baseline"]:.2f}')
    ax2.axhline(0.5, color='gray',
                linestyle=':', linewidth=1,
                label='50% (coin flip)')
    ax2.axhline(res['accuracy'], color='#2ecc71',
                linestyle='-', linewidth=1.2, alpha=0.7,
                label=f'Test acc {res["accuracy"]:.2f}')
    ax2.set_ylabel('Validation Accuracy', fontsize=8)
    ax2.set_xlabel('Epoch', fontsize=8)
    ax2.legend(fontsize=6, loc='lower right')
    ax2.grid(True, alpha=0.3)
    ax2.tick_params(labelsize=7)
    ax2.set_ylim(0.3, 0.9)

# Hide unused subplots
for j in range(n_sectors, n_rows * n_cols):
    axes[(j // n_cols) * 2, j % n_cols].set_visible(False)
    axes[(j // n_cols) * 2 + 1, j % n_cols].set_visible(False)

fig.suptitle(
    'LSTM Training Curves — Direct Sector Direction Prediction\n'
    'Top row = training loss (lower = better) | '
    'Bottom row = validation accuracy (higher = better)\n'
    'Red dashed = majority baseline | Gray dotted = 50% coin flip | '
    'Green = final test accuracy',
    fontsize=11, fontweight='bold'
)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/lstm_training_curves.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('✓ Saved lstm_training_curves.png')

In [ ]:
# ── CELL 10b: RNN training curves ─────────────────────────────
n_sectors = len(rnn_results)
n_cols    = 4
n_rows    = int(np.ceil(n_sectors / n_cols))

fig, axes = plt.subplots(n_rows * 2, n_cols,
                          figsize=(n_cols * 5, n_rows * 8))

for i, (ticker, res) in enumerate(rnn_results.items()):
    row_loss = (i // n_cols) * 2
    row_acc  = row_loss + 1
    col      = i % n_cols

    ax1 = axes[row_loss, col]
    ax1.plot(res['train_losses'], color='#c0392b', linewidth=1.5)
    ax1.set_title(f'{ticker} — {SECTOR_ETFS.get(ticker,"")}',
                  fontsize=9, fontweight='bold')
    ax1.set_ylabel('Training Loss', fontsize=8)
    ax1.set_xlabel('Epoch', fontsize=8)
    ax1.grid(True, alpha=0.3)
    ax1.tick_params(labelsize=7)

    ax2 = axes[row_acc, col]
    ax2.plot(res['val_accs'], color='#16a085', linewidth=1.5)
    ax2.axhline(res['baseline'], color='#e74c3c',
                linestyle='--', linewidth=1.2,
                label=f'Baseline {res["baseline"]:.2f}')
    ax2.axhline(0.5, color='gray',
                linestyle=':', linewidth=1,
                label='50% (coin flip)')
    ax2.axhline(res['accuracy'], color='#2ecc71',
                linestyle='-', linewidth=1.2, alpha=0.7,
                label=f'Test acc {res["accuracy"]:.2f}')
    ax2.set_ylabel('Validation Accuracy', fontsize=8)
    ax2.set_xlabel('Epoch', fontsize=8)
    ax2.legend(fontsize=6, loc='lower right')
    ax2.grid(True, alpha=0.3)
    ax2.tick_params(labelsize=7)
    ax2.set_ylim(0.3, 0.9)

for j in range(n_sectors, n_rows * n_cols):
    axes[(j // n_cols) * 2, j % n_cols].set_visible(False)
    axes[(j // n_cols) * 2 + 1, j % n_cols].set_visible(False)

fig.suptitle(
    'Vanilla RNN Training Curves — Direct Sector Direction Prediction',
    fontsize=11, fontweight='bold'
)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/rnn_training_curves.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('✓ Saved rnn_training_curves.png')


In [ ]:
# ── CELL 11: Baseline models for comparison ───────────────────
# Train Logistic Regression and Random Forest on same data
# so we can compare LSTM vs simpler models

rf_results = {}
lr_results = {}

print('=== Baseline Models: LR + Random Forest ===')
print(f'{"Sector":<8} {"LR Acc":>8} {"RF Acc":>8} {"LSTM Acc":>10} {"RNN Acc":>10} {"Baseline":>10}')
print('-' * 62)

for ticker in sector_cols:
    y_t = y_dir[ticker].iloc[:train_n].values.astype(float)
    y_e = y_dir[ticker].iloc[train_n:].values.astype(float)

    vtr = ~np.isnan(y_t);  vte = ~np.isnan(y_e)
    y_t, y_e = y_t[vtr].astype(int), y_e[vte].astype(int)
    Xtr_b, Xte_b = X_train_s[vtr], X_test_s[vte]

    if len(y_t) < 20 or len(y_e) < 5:
        continue

    # Logistic Regression
    lr = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
    lr.fit(Xtr_b, y_t)
    acc_lr = accuracy_score(y_e, lr.predict(Xte_b))
    lr_results[ticker] = {
        'pred':  lr.predict(Xte_b),
        'probs': lr.predict_proba(Xte_b)[:, 1],
        'acc':   acc_lr,
        'dates': X_test.index[vte]
    }

    # Random Forest
    rf = RandomForestClassifier(n_estimators=200, max_depth=6,
                                 random_state=RANDOM_STATE, n_jobs=-1)
    rf.fit(Xtr_b, y_t)
    acc_rf = accuracy_score(y_e, rf.predict(Xte_b))
    rf_results[ticker] = {
        'pred':  rf.predict(Xte_b),
        'probs': rf.predict_proba(Xte_b)[:, 1],
        'acc':   acc_rf,
        'dates': X_test.index[vte]
    }

    lstm_acc = lstm_results.get(ticker, {}).get('accuracy', float('nan'))
    rnn_acc  = rnn_results.get(ticker, {}).get('accuracy', float('nan'))
    baseline = max(y_t.mean(), 1 - y_t.mean())
    print(f'{ticker:<8} {acc_lr:>8.3f} {acc_rf:>8.3f} '
          f'{lstm_acc:>10.3f} {rnn_acc:>10.3f} {baseline:>10.3f}')


In [ ]:
# ── CELL 12: Accuracy comparison chart ────────────────────────
tickers_plot = [t for t in sector_cols
              if t in lstm_results and t in rnn_results]

lr_accs   = [lr_results.get(t, {}).get('acc', 0)   for t in tickers_plot]
rf_accs   = [rf_results.get(t, {}).get('acc', 0)   for t in tickers_plot]
lstm_accs = [lstm_results[t]['accuracy']            for t in tickers_plot]
rnn_accs  = [rnn_results[t]['accuracy']            for t in tickers_plot]
baselines = [lstm_results[t]['baseline']            for t in tickers_plot]

x     = np.arange(len(tickers_plot))
width = 0.2

fig, ax = plt.subplots(figsize=(18, 7))

b1 = ax.bar(x - 1.5 * width, lr_accs,   width, label='Logistic Regression',
            color='#9b59b6', alpha=0.85, edgecolor='white')
b2 = ax.bar(x - 0.5 * width, rf_accs,   width, label='Random Forest',
            color='#3498db', alpha=0.85, edgecolor='white')
b3 = ax.bar(x + 0.5 * width, lstm_accs, width, label='LSTM',
            color='#e74c3c', alpha=0.85, edgecolor='white')
b4 = ax.bar(x + 1.5 * width, rnn_accs,  width, label='Vanilla RNN',
            color='#16a085', alpha=0.85, edgecolor='white')

# Baseline as scatter dots
ax.scatter(x, baselines, color='black', s=60, zorder=5,
           label='Majority baseline', marker='D')
ax.axhline(0.5, color='gray', linestyle=':', linewidth=1.2,
           alpha=0.7, label='50% (coin flip)')

ax.set_xticks(x)
ax.set_xticklabels(tickers_plot, fontsize=10)
ax.set_ylabel('Test Accuracy', fontsize=12)
ax.set_ylim(0.3, 0.85)
ax.set_title(
    'Sector Direction Prediction Accuracy: LR vs Random Forest vs LSTM\n'
    '(black diamond = majority class baseline, dotted = 50% coin flip)',
    fontsize=13, fontweight='bold'
)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)

# Value labels on top of each bar
for bars in [b1, b2, b3, b4]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.005,
                f'{h:.2f}', ha='center', va='bottom', fontsize=7)

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/sequence_models_accuracy_comparison.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('✓ Saved sequence_models_accuracy_comparison.png')


In [ ]:
# ── CELL 13: Portfolio metrics helpers ────────────────────────
def total_ret(r):
    return ((1 + r).prod() - 1) * 100

def ann_ret(r):
    n = len(r)
    return ((1 + r).prod() ** (12/n) - 1) * 100 if n > 0 else 0

def sharpe(r, rf=RISK_FREE_RATE):
    ex = r - rf/12
    return (ex.mean() / ex.std()) * np.sqrt(12) if ex.std() > 0 else 0

def max_dd(r):
    w = (1 + r).cumprod()
    return ((w - w.cummax()) / w.cummax()).min() * 100

def win_rate(s, b):
    a = pd.DataFrame({'s': s, 'b': b}).dropna()
    return (a['s'] > a['b']).mean() * 100

def get_metrics(r, bm, name):
    b = bm.reindex(r.index).fillna(0)
    return {
        'Strategy':      name,
        'Total Ret (%)': round(total_ret(r), 2),
        'Ann Ret (%)':   round(ann_ret(r), 2),
        'Sharpe':        round(sharpe(r), 3),
        'Max DD (%)':    round(max_dd(r), 2),
        'Win Rate (%)':  round(win_rate(r, b), 1),
        'Volatility (%)': round(r.std() * np.sqrt(12) * 100, 2),
        'N Months':      len(r),
    }

print('✓ Metrics helpers defined')

In [ ]:
# ── CELL 14: Backtest function ─────────────────────────────────
def backtest_direct(direct_res, sect_rets, top_n=TOP_N_SECTORS,
                    cost=TRANSACTION_COST):
    """
    For each month T where we have predictions:
      1. Look at predicted probability of going UP for each sector
      2. Invest equally in top-N sectors by probability
      3. Collect actual returns from month T+1
    """
    # Build probability DataFrame indexed by date
    prob_frames = {}
    for ticker, res in direct_res.items():
        dates = res['dates']
        probs = res['probabilities']
        n     = min(len(dates), len(probs))
        prob_frames[ticker] = pd.Series(probs[:n], index=dates[:n])

    prob_df = pd.DataFrame(prob_frames).dropna(how='all')

    monthly_rets = []
    dates_out    = []

    for t_date in prob_df.index:
        row  = prob_df.loc[t_date].dropna()
        if len(row) == 0:
            continue

        # Top-N by predicted probability of going up
        top  = row.nlargest(top_n).index.tolist()

        # Get next month's actual returns
        fut  = sect_rets.index[sect_rets.index > t_date]
        if len(fut) == 0:
            continue
        next_d = fut[0]

        row_r  = sect_rets.loc[next_d]
        avail  = [s for s in top if s in row_r.index
                  and not pd.isna(row_r[s])]
        if not avail:
            continue

        ret = row_r[avail].mean() - cost * len(avail)
        monthly_rets.append(ret)
        dates_out.append(next_d)

    return pd.Series(monthly_rets, index=pd.DatetimeIndex(dates_out))


def backtest_regime(predicted_regimes, top_sectors_dict,
                    sect_rets, cost=TRANSACTION_COST):
    """
    For each month T with a predicted regime:
      1. Look up top sectors for that regime
      2. Equal-weight invest in those sectors
      3. Collect next month's actual returns
    """
    monthly_rets = []
    dates_out    = []

    for t_date in predicted_regimes.index:
        regime   = predicted_regimes[t_date]
        selected = top_sectors_dict.get(regime, [])

        fut = sect_rets.index[sect_rets.index > t_date]
        if len(fut) == 0:
            continue
        next_d = fut[0]

        row   = sect_rets.loc[next_d]
        avail = [s for s in selected
                 if s in row.index and not pd.isna(row[s])]
        if not avail:
            continue

        ret = row[avail].mean() - cost * len(avail)
        monthly_rets.append(ret)
        dates_out.append(next_d)

    return pd.Series(monthly_rets, index=pd.DatetimeIndex(dates_out))

print('✓ Backtest functions defined')

In [ ]:
# ── CELL 15: Run all backtests ─────────────────────────────────
import json

# Load regime top sectors from notebook 01
# Make sure you saved this in notebook 01:
#   import json
#   with open(f'{DATA_DIR}/regime_top_sectors.json','w') as f:
#       json.dump(regime_top_sectors, f)
with open(f'{DATA_DIR}/regime_top_sectors.json') as f:
    regime_top_sectors = json.load(f)

print('Loaded regime top sectors:')
for regime, sectors in regime_top_sectors.items():
    print(f'  {regime}: {sectors}')

# Test period benchmark
bm_clean = bm_test.dropna()

all_strategies = {}

# ── Strategy 1: LSTM direct prediction ────────────────────────
strat_lstm = backtest_direct(lstm_results, returns_df[sector_cols])
all_strategies['LSTM Direct'] = strat_lstm
print(f'\nLSTM Direct: {len(strat_lstm)} months backtested')

strat_rnn = backtest_direct(rnn_results, returns_df[sector_cols])
all_strategies['RNN Direct'] = strat_rnn
print(f'RNN Direct:  {len(strat_rnn)} months backtested')

# ── Strategy 2: Random Forest direct prediction ────────────────
strat_rf = backtest_direct(rf_results, returns_df[sector_cols])
all_strategies['Random Forest Direct'] = strat_rf
print(f'RF Direct:   {len(strat_rf)} months backtested')

# ── Strategy 3: Logistic Regression direct prediction ─────────
strat_lr = backtest_direct(lr_results, returns_df[sector_cols])
all_strategies['Logistic Regression Direct'] = strat_lr
print(f'LR Direct:   {len(strat_lr)} months backtested')

# ── Strategy 4: Macro regime rotation (from notebook 01) ──────
# Use regime labels directly on test period as "perfect" regime knowledge
test_regimes = regime_labels.reindex(X_test.index).dropna()
strat_regime = backtest_regime(test_regimes, regime_top_sectors,
                                returns_df[sector_cols])
all_strategies['Macro Regime Rotation'] = strat_regime
print(f'Regime:      {len(strat_regime)} months backtested')

# ── Strategy 5: Equal weight all sectors (naive baseline) ──────
strat_ew = returns_df[sector_cols].reindex(bm_clean.index).mean(axis=1)
all_strategies['Equal Weight Sectors'] = strat_ew
print(f'Equal Weight: {len(strat_ew)} months')

print('\n✓ All backtests complete')


In [ ]:
# ── CELL 16: Performance metrics table ────────────────────────
rows = []
for name, r in all_strategies.items():
    if len(r) < 5:
        continue
    rows.append(get_metrics(r, bm_clean, name))

# Add SPY benchmark
rows.append(get_metrics(bm_clean, bm_clean, 'SPY Benchmark'))

results_table = pd.DataFrame(rows).set_index('Strategy')

print('=' * 75)
print('FINAL PERFORMANCE COMPARISON — TEST PERIOD')
print('=' * 75)
print(results_table.to_string())
print()
print('Key metrics explained:')
print('  Sharpe    = risk-adjusted return (higher = better, >1 is good)')
print('  Max DD    = worst peak-to-trough loss (less negative = better)')
print('  Win Rate  = % of months strategy beats SPY')

results_table.to_csv(f'{RESULTS_DIR}/lstm_backtest_results.csv')
print(f'\n✓ Saved to {RESULTS_DIR}/lstm_backtest_results.csv')

In [ ]:
# ── CELL 17: Cumulative returns chart ─────────────────────────
STRAT_COLORS = {
    'LSTM Direct':                 '#e74c3c',
    'RNN Direct':                  '#16a085',
    'Random Forest Direct':        '#3498db',
    'Logistic Regression Direct':  '#9b59b6',
    'Macro Regime Rotation':       '#2ecc71',
    'Equal Weight Sectors':        '#f39c12',
    'SPY Benchmark':               'black',
}
STRAT_STYLES = {
    'LSTM Direct':                 '-',
    'RNN Direct':                  '-',
    'Random Forest Direct':        '--',
    'Logistic Regression Direct':  '-.',
    'Macro Regime Rotation':       '-',
    'Equal Weight Sectors':        ':',
    'SPY Benchmark':               '-',
}

fig, ax = plt.subplots(figsize=(18, 9))

# Regime background shading
test_regime_plot = regime_labels.reindex(bm_clean.index).dropna()
prev_r, start_d  = None, None
for date, regime in test_regime_plot.items():
    if regime != prev_r:
        if prev_r is not None:
            ax.axvspan(start_d, date, alpha=0.10,
                       color=REGIME_COLORS[prev_r], zorder=1)
        start_d, prev_r = date, regime
if prev_r:
    ax.axvspan(start_d, bm_clean.index[-1], alpha=0.10,
               color=REGIME_COLORS[prev_r], zorder=1)

# SPY benchmark — thick black
bm_cum = (1 + bm_clean).cumprod() * 100
ax.plot(bm_cum.index, bm_cum.values,
        color='black', linewidth=3, zorder=6, label='SPY Benchmark')

# All strategies
for name, r in all_strategies.items():
    if len(r) < 5:
        continue
    r_aligned = r.reindex(bm_clean.index).fillna(0)
    cum = (1 + r_aligned).cumprod() * 100
    ax.plot(cum.index, cum.values,
            color=STRAT_COLORS.get(name, 'gray'),
            linestyle=STRAT_STYLES.get(name, '-'),
            linewidth=2.2, alpha=0.9, label=name, zorder=4)

# Regime legend
regime_patches = [
    mpatches.Patch(color=c, alpha=0.4, label=r)
    for r, c in REGIME_COLORS.items()
]
regime_leg = ax.legend(
    handles=regime_patches, loc='upper left',
    fontsize=9, title='Macro Regime (background)',
    title_fontsize=9, framealpha=0.9
)
ax.add_artist(regime_leg)

# Strategy legend
ax.legend(loc='upper center', bbox_to_anchor=(0.5, 0.99),
          ncol=3, fontsize=9, framealpha=0.9,
          title='Strategies', title_fontsize=9)

ax.yaxis.set_major_formatter(
    mtick.FuncFormatter(lambda y, _: f'{y:.0f}')
)
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Portfolio Value (start = 100)', fontsize=12)
ax.set_title(
    'Cumulative Returns: LSTM vs Regime Rotation vs SPY (Test Period)\n'
    'Background = Macroeconomic Regime | Start = 100',
    fontsize=14, fontweight='bold'
)
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/lstm_cumulative_returns.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('✓ Saved lstm_cumulative_returns.png')


In [ ]:
# ── CELL 18: Sharpe ratio bar chart ───────────────────────────
strategies_to_plot = list(all_strategies.keys()) + ['SPY Benchmark']
all_with_spy = {**all_strategies, 'SPY Benchmark': bm_clean}

sharpes = []
for name in strategies_to_plot:
    r = all_with_spy.get(name, pd.Series())
    if len(r) > 5:
        sharpes.append({'Strategy': name, 'Sharpe': sharpe(r)})

sharpe_df = pd.DataFrame(sharpes).set_index('Strategy').sort_values(
    'Sharpe', ascending=True
)

colors_bar = []
for s, v in sharpe_df['Sharpe'].items():
    if v < 0:
        colors_bar.append('#e74c3c')
    elif s == 'LSTM Direct':
        colors_bar.append('#2ecc71')
    elif s == 'RNN Direct':
        colors_bar.append('#1abc9c')
    else:
        colors_bar.append('#3498db')

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(sharpe_df.index, sharpe_df['Sharpe'],
               color=colors_bar, edgecolor='white', linewidth=1.2, alpha=0.85)
ax.axvline(0, color='black', linewidth=0.8)

spy_sharpe = sharpe(bm_clean)
ax.axvline(spy_sharpe, color='black', linewidth=2,
           linestyle='--', alpha=0.7, label=f'SPY Sharpe = {spy_sharpe:.3f}')

for bar, val in zip(bars, sharpe_df['Sharpe']):
    xpos = val + 0.01 if val >= 0 else val - 0.01
    ha   = 'left'     if val >= 0 else 'right'
    ax.text(xpos, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', ha=ha, fontsize=10, fontweight='bold')

ax.set_xlabel('Annualized Sharpe Ratio', fontsize=12)
ax.set_title(
    'Sharpe Ratio Comparison — All Strategies\n'
    '(higher = better risk-adjusted return | dashed = SPY benchmark)',
    fontsize=13, fontweight='bold'
)
ax.legend(fontsize=10)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/lstm_sharpe_comparison.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('✓ Saved lstm_sharpe_comparison.png')


In [ ]:
# ── CELL 19: Monthly return heatmap (calendar view) ───────────
# Shows which months each strategy did well — useful for spotting
# whether LSTM wins in specific macro regimes

def monthly_heatmap(returns_series, title, ax):
    r = returns_series.copy()
    r.index = pd.to_datetime(r.index)
    pivot = r.groupby([r.index.year, r.index.month]).mean().unstack()
    pivot.columns = ['Jan','Feb','Mar','Apr','May','Jun',
                     'Jul','Aug','Sep','Oct','Nov','Dec']
    sns.heatmap(pivot * 100, annot=True, fmt='.1f',
                cmap='RdYlGn', center=0, ax=ax,
                linewidths=0.3, cbar=False,
                annot_kws={'fontsize': 7})
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.set_ylabel('Year')
    ax.tick_params(labelsize=8)

fig, axes = plt.subplots(4, 1, figsize=(16, 22))

monthly_heatmap(all_strategies['LSTM Direct'],
                'LSTM Direct — Monthly Returns (%)', axes[0])
monthly_heatmap(all_strategies['RNN Direct'],
                'Vanilla RNN Direct — Monthly Returns (%)', axes[1])
monthly_heatmap(all_strategies['Macro Regime Rotation'],
                'Macro Regime Rotation — Monthly Returns (%)', axes[2])
monthly_heatmap(bm_clean,
                'SPY Benchmark — Monthly Returns (%)', axes[3])

plt.suptitle(
    'Monthly Return Calendar Heatmap\n'
    'Green = positive month, Red = negative month',
    fontsize=13, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/lstm_monthly_heatmap.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('✓ Saved lstm_monthly_heatmap.png')


In [ ]:
# ── CELL 20: Predicted probabilities over time (LSTM then RNN) ─
# Shows what the LSTM believed about each sector over the test period
# High probability = LSTM was confident sector would go up

prob_frames = {}
for ticker, res in lstm_results.items():
    dates = res['dates']
    probs = res['probabilities']
    n     = min(len(dates), len(probs))
    prob_frames[ticker] = pd.Series(probs[:n], index=dates[:n])

prob_df = pd.DataFrame(prob_frames)

fig, ax = plt.subplots(figsize=(18, 8))

# Regime background
prev_r, start_d = None, None
for date, regime in test_regime_plot.items():
    if regime != prev_r:
        if prev_r:
            ax.axvspan(start_d, date, alpha=0.10,
                       color=REGIME_COLORS[prev_r], zorder=1)
        start_d, prev_r = date, regime
if prev_r:
    ax.axvspan(start_d, prob_df.index[-1], alpha=0.10,
               color=REGIME_COLORS[prev_r], zorder=1)

SECTOR_COLORS = {
    'XLK':'#e74c3c', 'XLF':'#3498db', 'XLE':'#f39c12',
    'XLV':'#2ecc71', 'XLI':'#9b59b6', 'XLP':'#1abc9c',
    'XLY':'#e67e22', 'XLU':'#34495e', 'XLB':'#e91e63',
    'XLRE':'#795548','XLC':'#607d8b',
}

for ticker in prob_df.columns:
    ax.plot(prob_df.index, prob_df[ticker],
            color=SECTOR_COLORS.get(ticker, 'gray'),
            linewidth=1.2, alpha=0.7,
            label=f'{ticker}')

ax.axhline(0.5, color='black', linewidth=1.5,
           linestyle='--', alpha=0.6, label='0.5 threshold')

regime_patches = [
    mpatches.Patch(color=c, alpha=0.4, label=r)
    for r, c in REGIME_COLORS.items()
]
regime_leg = ax.legend(
    handles=regime_patches, loc='lower left',
    fontsize=8, title='Macro Regime', title_fontsize=8
)
ax.add_artist(regime_leg)
ax.legend(loc='upper right', ncol=4, fontsize=8,
          title='Sectors', title_fontsize=8)

ax.set_ylim(0, 1)
ax.set_ylabel('LSTM Predicted Probability (sector goes UP)', fontsize=11)
ax.set_xlabel('Date', fontsize=11)
ax.set_title(
    'LSTM Predicted Probabilities Over Time\n'
    'Above 0.5 = LSTM predicts sector goes UP | '
    'Below 0.5 = predicts DOWN\n'
    'Background = actual macro regime during that period',
    fontsize=12, fontweight='bold'
)
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/lstm_probabilities_over_time.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('✓ Saved lstm_probabilities_over_time.png')

# ── Vanilla RNN probabilities ───────────────────────────────────
prob_frames_r = {}
for ticker, res in rnn_results.items():
    dates = res['dates']
    probs = res['probabilities']
    n     = min(len(dates), len(probs))
    prob_frames_r[ticker] = pd.Series(probs[:n], index=dates[:n])

prob_df_r = pd.DataFrame(prob_frames_r)

fig, ax = plt.subplots(figsize=(18, 8))
prev_r, start_d = None, None
for date, regime in test_regime_plot.items():
    if regime != prev_r:
        if prev_r:
            ax.axvspan(start_d, date, alpha=0.10,
                       color=REGIME_COLORS[prev_r], zorder=1)
        start_d, prev_r = date, regime
if prev_r:
    ax.axvspan(start_d, prob_df_r.index[-1], alpha=0.10,
               color=REGIME_COLORS[prev_r], zorder=1)

for ticker in prob_df_r.columns:
    ax.plot(prob_df_r.index, prob_df_r[ticker],
            color=SECTOR_COLORS.get(ticker, 'gray'),
            linewidth=1.2, alpha=0.7,
            label=f'{ticker}')

ax.axhline(0.5, color='black', linewidth=1.5,
           linestyle='--', alpha=0.6, label='0.5 threshold')

regime_patches = [
    mpatches.Patch(color=c, alpha=0.4, label=r)
    for r, c in REGIME_COLORS.items()
]
regime_leg = ax.legend(
    handles=regime_patches, loc='lower left',
    fontsize=8, title='Macro Regime', title_fontsize=8
)
ax.add_artist(regime_leg)
ax.legend(loc='upper right', ncol=4, fontsize=8,
          title='Sectors', title_fontsize=8)

ax.set_ylim(0, 1)
ax.set_ylabel('RNN predicted P(up)', fontsize=11)
ax.set_xlabel('Date', fontsize=11)
ax.set_title(
    'Vanilla RNN — Predicted Probabilities Over Time',
    fontsize=12, fontweight='bold'
)
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/rnn_probabilities_over_time.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('✓ Saved rnn_probabilities_over_time.png')


In [ ]:
# ── CELL 21: Regime-level accuracy (LSTM vs RNN) ──────────────


def regime_accuracy_report(results_dict, model_name, fname_suffix):
    print(f'=== {model_name} Accuracy by Macro Regime ===')
    print('Does the model find it easier to predict in some regimes?')
    print()

    regime_accuracy_data = []
    for ticker, res in results_dict.items():
        dates  = res['dates']
        preds  = res['predictions']
        y_true = res['y_true']
        n      = min(len(dates), len(preds), len(y_true))

        df_ticker = pd.DataFrame({
            'pred':    preds[:n],
            'actual':  y_true[:n],
            'correct': (preds[:n] == y_true[:n]).astype(int),
        }, index=dates[:n])

        df_ticker = df_ticker.join(
            regime_labels.rename('regime'), how='left'
        ).dropna(subset=['regime'])

        for regime, grp in df_ticker.groupby('regime'):
            if len(grp) < 3:
                continue
            regime_accuracy_data.append({
                'Ticker': ticker,
                'Regime': regime,
                'Accuracy': grp['correct'].mean(),
                'N Months': len(grp),
            })

    regime_acc_df = pd.DataFrame(regime_accuracy_data)
    pivot_acc = regime_acc_df.pivot_table(
        index='Regime', columns='Ticker',
        values='Accuracy', aggfunc='mean'
    )

    print(pivot_acc.round(3).to_string())
    print()
    print('Mean accuracy per regime (averaged across sectors):')
    print(pivot_acc.mean(axis=1).round(3).to_string())

    fig, ax = plt.subplots(figsize=(14, 5))
    sns.heatmap(
        pivot_acc, annot=True, fmt='.2f',
        cmap='RdYlGn', center=0.5,
        vmin=0.3, vmax=0.8,
        ax=ax, linewidths=0.5,
        cbar_kws={'label': 'Prediction Accuracy'}
    )
    ax.set_title(
        f'{model_name} Prediction Accuracy by Macro Regime\n'
        '(green = high accuracy, red = low accuracy | 0.5 = coin flip)',
        fontsize=12, fontweight='bold'
    )
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.savefig(f'{RESULTS_DIR}/{fname_suffix}',
                dpi=150, bbox_inches='tight')
    plt.show()
    print(f'✓ Saved {fname_suffix}')


regime_accuracy_report(lstm_results, 'LSTM', 'lstm_accuracy_by_regime.png')
regime_accuracy_report(rnn_results, 'Vanilla RNN', 'rnn_accuracy_by_regime.png')


In [ ]:
# ── CELL 22: Final summary and key findings ────────────────────
print('=' * 70)
print('FINAL SUMMARY — LSTM & RNN DIRECT PREDICTION NOTEBOOK')
print('=' * 70)
print()
print('PERFORMANCE TABLE:')
print(results_table.to_string())
print()

best_sharpe = results_table['Sharpe'].idxmax()
best_return = results_table['Total Ret (%)'].idxmax()
best_dd     = results_table['Max DD (%)'].idxmax()

print(f'Best Sharpe ratio:      {best_sharpe} ({results_table.loc[best_sharpe, "Sharpe"]:.3f})')
print(f'Best total return:      {best_return} ({results_table.loc[best_return, "Total Ret (%)"]:.1f}%)')
print(f'Smallest max drawdown:  {best_dd} ({results_table.loc[best_dd, "Max DD (%)"]:.1f}%)')

print()
print('LSTM vs BASELINE SUMMARY:')
beats = sum(1 for r in lstm_results.values() if r['accuracy'] > r['baseline'])
total = len(lstm_results)
print(f'  LSTM beats majority baseline in {beats}/{total} sectors')
avg_acc  = np.mean([r['accuracy']  for r in lstm_results.values()])
avg_base = np.mean([r['baseline']  for r in lstm_results.values()])
print(f'  Average LSTM accuracy:  {avg_acc:.3f}')
print(f'  Average baseline:       {avg_base:.3f}')
print(f'  Average improvement:    {avg_acc - avg_base:+.3f}')

print()
print('RNN vs BASELINE SUMMARY:')
beats_r = sum(1 for r in rnn_results.values() if r['accuracy'] > r['baseline'])
total_r = len(rnn_results)
print(f'  RNN beats majority baseline in {beats_r}/{total_r} sectors')
avg_acc_r  = np.mean([r['accuracy']  for r in rnn_results.values()])
avg_base_r = np.mean([r['baseline']  for r in rnn_results.values()])
print(f'  Average RNN accuracy:   {avg_acc_r:.3f}')
print(f'  Average baseline:       {avg_base_r:.3f}')
print(f'  Average improvement:    {avg_acc_r - avg_base_r:+.3f}')

print()
print('OUTPUT FILES SAVED:')
outputs = [
    'lstm_training_curves.png',
    'rnn_training_curves.png',
    'sequence_models_accuracy_comparison.png',
    'lstm_cumulative_returns.png',
    'lstm_sharpe_comparison.png',
    'lstm_monthly_heatmap.png',
    'lstm_probabilities_over_time.png',
    'rnn_probabilities_over_time.png',
    'lstm_accuracy_by_regime.png',
    'rnn_accuracy_by_regime.png',
    'lstm_backtest_results.csv',
]
for o in outputs:
    print(f'  {o}')
